# The clair Python API

Each command of the clair CLI is one function of the `clair` package. The CLI parses the
arguments and it calls these functions. Thus a notebook does the same work, and it makes
no `subprocess` call.

| Function | The equivalent command | Needs a warehouse |
|----------|------------------------|-------------------|
| `clair.compile()` | `clair compile` | no |
| `clair.validate()` | `clair validate` | no |
| `clair.catalog()` | the data behind `clair docs` | no |
| `clair.clean()` | `clair clean` | no |
| `clair.run()` | `clair run` | yes |
| `clair.test()` | `clair test` | yes |

This notebook uses the four functions that need no warehouse.
[03_run_without_snowflake.ipynb](03_run_without_snowflake.ipynb) shows `clair.run()`.

Each function gives a result object with the complete data of the operation. No function
writes to stdout, and no function stops the process. A fault raises a `ClairError`.

In [1]:
from pathlib import Path


def find_examples_root(start: Path | None = None) -> Path:
    """Give the directory that holds examples/projects.

    The search starts at the working directory and it goes up. Thus the
    notebook runs from this directory, from the repository root, or from a
    Jupyter server that you started at a different place.
    """
    for directory in [start or Path.cwd(), *(start or Path.cwd()).parents]:
        if (directory / "examples" / "projects").is_dir():
            return directory
    raise FileNotFoundError("No parent directory holds examples/projects.")


REPOSITORY_ROOT = find_examples_root()

# The CLI configures structlog; a notebook does not. This line keeps the INFO
# messages of each operation out of the cells. Remove it to read them.
import logging

import structlog

structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import os

import pandas as pd

import clair

PROJECT = REPOSITORY_ROOT / "examples" / "projects" / "example_4"

# The routing entry of the dev environment reads CLAIR_USER, and it puts each
# Trouve in a database of that name. See examples/projects/example_4/__routing__.py.
os.environ["CLAIR_USER"] = "notebook_user"

print("clair", clair.__version__, "-- project", PROJECT.name)

clair 0.1.0 -- project example_4


## Two environments, in a temporary file

An environment holds the connection settings, and it gives the routing entry its name.
clair reads `~/.clair/environments.yml`. This notebook must not depend on your file, thus
it writes its own and it points clair at it. Your project needs this cell in no place: run
`clair init` one time, and clair reads your file.

In [2]:
import tempfile
from pathlib import Path

import clair.environments.environments as environments_module

NOTEBOOK_DIRECTORY = Path(tempfile.mkdtemp(prefix="clair-notebook-"))
environments_file = NOTEBOOK_DIRECTORY / "environments.yml"
environments_file.write_text(
    """
dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4
""".strip()
)

# clair reads ~/.clair/environments.yml. This line points it at the file above.
environments_module.DEFAULT_ENVIRONMENTS_PATH = environments_file

print(environments_file.read_text())

dev:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 1

prod:
  account: notebook
  user: notebook_user
  warehouse: notebook_warehouse
  role: NOTEBOOK
  password: unused-because-this-notebook-opens-no-connection
  threads: 4


## `clair.compile()` — the SQL, with no connection

Compilation resolves each reference between the Trouve files, it applies the routing entry
of the environment, and it writes the statements to the artifacts directory. It opens no
connection. Thus it is the fastest way to see what a run will do.

In [3]:
output = clair.compile(PROJECT, env="prod")

print(f"{output.trouve_count} Trouves, {output.source_count} sources")
print(f"artifacts: {output.artifacts_dir}")

pd.DataFrame(
    [
        {
            "order": index,
            "logical address": node.logical_address,
            "physical address": node.physical_address,
            "type": node.type,
            "engine": node.execution_type.value,
            "reads": ", ".join(dependency.split(".")[-1] for dependency in node.dependencies),
            "statements": len(node.sql),
        }
        for index, node in enumerate(output.compiled_nodes, start=1)
    ]
)

4 Trouves, 1 sources
artifacts: /Users/Omer/Documents/Nerd/Rivage/clair/.claude/worktrees/notebooks-refresh/examples/projects/example_4/_clairtifacts/01a04a8ff2567c57b53b6020a457bf5f


,order,logical address,physical address,type,engine,reads,statements
0,1,example_4_database.reference.event_type_labels,example_4_database.reference.event_type_labels,TABLE,pandas,,0
1,2,example_4_database.refined.events,example_4_database.refined.events,TABLE,snowflake,events,4
2,3,example_4_database.derived.daily_event_counts,example_4_database.derived.daily_event_counts,TABLE,pandas,events,0
3,4,example_4_database.derived.top_event_types,example_4_database.derived.top_event_types,TABLE,snowflake,"daily_event_counts, event_type_labels",4


The `engine` column separates the two execution types. A `Trouve` with a `sql` string runs
in Snowflake. A `PandasTrouve` and a `SeedTrouve` run on this machine, and they have no
statement: clair writes the DataFrame to the warehouse.

`output.node(address)` finds one node by its logical address or its physical address.

In [4]:
node = output.node("example_4_database.refined.events")
print("\n\n".join(node.sql))

CREATE OR REPLACE TABLE example_4_database.refined.events__clair_01a04a8ff2567c57b53b6020a457bf5f AS (
select
            event_id,
            user_id,
            event_type,
            occurred_at,
            occurred_at::date                   as event_date,

            -- page_view / button_click
            properties:page::string             as page,
            properties:referrer::string         as referrer,

            -- button_click
            properties:element::string          as element,

            -- form_submit
            properties:form::string             as form,
            properties:success::boolean         as form_success,

            -- purchase
            properties:amount::float            as purchase_amount,
            properties:currency::string         as purchase_currency,
            properties:item_id::string          as purchase_item_id
        from example_4_database.source.events
)

-- staging: the data quality tests run here

-- staging: 

## The routing entry decides the physical address

The logical address comes from the file path. The physical address comes from the routing
entry of the active environment. Compile the project against two environments to see the
difference. This is how one person builds a private copy of the complete project.

In [5]:
addresses_of_environment = {}
for environment_name in ("prod", "dev"):
    compile_output = clair.compile(PROJECT, env=environment_name)
    addresses_of_environment[environment_name] = {
        node.logical_address: node.physical_address for node in compile_output.compiled_nodes
    }

pd.DataFrame(
    [
        {
            "logical address": logical_address,
            "prod writes to": physical_address,
            "dev writes to": addresses_of_environment["dev"][logical_address],
        }
        for logical_address, physical_address in addresses_of_environment["prod"].items()
    ]
)

,logical address,prod writes to,dev writes to
0,example_4_database.reference.event_type_labels,example_4_database.reference.event_type_labels,notebook_user.reference.event_type_labels
1,example_4_database.refined.events,example_4_database.refined.events,notebook_user.refined.events
2,example_4_database.derived.daily_event_counts,example_4_database.derived.daily_event_counts,notebook_user.derived.daily_event_counts
3,example_4_database.derived.top_event_types,example_4_database.derived.top_event_types,notebook_user.derived.top_event_types


## Selectors — compile one part of the project

`select` and `exclude` take the patterns of `--select` and `--exclude`. The `+` operator
adds the neighbours from the DAG.

| Pattern | clair selects |
|---------|---------------|
| `pattern` | the matches only |
| `+pattern` | the matches, and each parent, at any distance |
| `pattern+` | the matches, and each child, at any distance |
| `1+pattern` | the matches, and the parents one level up |

In [6]:
target = "example_4_database.derived.top_event_types"
patterns = [target, f"+{target}", "example_4_database.refined.*+", f"1+{target}"]

for pattern in patterns:
    selected = clair.compile(PROJECT, env="prod", select=[pattern])
    names = [node.logical_address.split(".", 1)[1] for node in selected.compiled_nodes]
    print(f"{pattern:<48} {names}")

example_4_database.derived.top_event_types       ['derived.top_event_types']


+example_4_database.derived.top_event_types      ['reference.event_type_labels', 'refined.events', 'derived.daily_event_counts', 'derived.top_event_types']


example_4_database.refined.*+                    ['refined.events', 'derived.daily_event_counts', 'derived.top_event_types']


1+example_4_database.derived.top_event_types     ['reference.event_type_labels', 'derived.daily_event_counts', 'derived.top_event_types']


A source is never a build target, thus clair removes it from the selection.
`exclude` removes a Trouve after the selection:

In [7]:
selected = clair.compile(
    PROJECT,
    env="prod",
    select=["example_4_database.*.*"],
    exclude=["example_4_database.derived.*"],
)
print([node.logical_address for node in selected.compiled_nodes])

['example_4_database.reference.event_type_labels', 'example_4_database.refined.events']


## Run modes — the SQL of an incremental run

`run_mode` changes the statements that clair writes. A Trouve with a `RunConfig` that
holds an `IncrementalMode` gives a merge or an insert in place of a `create or replace`.
`effective_run_mode` tells you the mode that clair used after it read the `RunConfig` of
that Trouve.

In [8]:
from clair import RunMode

for run_mode in (RunMode.FULL_REFRESH, RunMode.INCREMENTAL):
    compile_output = clair.compile(PROJECT, env="prod", run_mode=run_mode)
    node = compile_output.node("example_4_database.refined.events")
    first_words = " ".join(node.sql[0].split()[:5])
    print(f"asked {run_mode.value:<14} used {node.effective_run_mode.value:<14} {first_words} ...")

asked full_refresh   used full_refresh   CREATE OR REPLACE TABLE example_4_database.refined.events__clair_01a04a8ffa6873ccb2459fd68a37e0cf ...


asked incremental    used full_refresh   CREATE OR REPLACE TABLE example_4_database.refined.events__clair_01a04a8ffb587e34ae1b7daa5c2b6b4e ...


The two modes give the same statement here, because no Trouve of `example_4` declares an
incremental config. [The incrementality topic](../../site_docs/docs/topics/incrementality.md)
shows a project that does.

## `clair.validate()` — find a routing fault before the SQL runs

Validation applies the routing entry to each Trouve and it reads the result. It finds an
address that Snowflake rejects, two Trouves that write to one address, and an address that
an author wrote as text in place of an f-string reference.

In [9]:
report = clair.validate(PROJECT, env="dev")

print(f"environment:    {report.env_name}")
print(f"routing entry:  {report.routing_description}")
print(f"routed Trouves: {report.routable_count}")
print(f"is_valid:       {report.is_valid}  (problems: {report.problem_count})")

for collision in report.collisions:
    print("collision:", collision.physical_address, collision.logical_addresses)
for problem in report.address_problems:
    print("address problem:", problem.logical_address)
for text_reference in report.text_references:
    print("text reference:", text_reference.logical_address)

environment:    dev
routing entry:  DeveloperRouting(environment_name='dev', user_variable='CLAIR_USER')
routed Trouves: 5
is_valid:       True  (problems: 0)


The report holds lists, thus your code reads the attributes and it parses no text.
`report.render()` gives the text that the CLI prints.

A validation loop over each environment is a good test for continuous integration:

In [10]:
for environment_name in ("dev", "prod"):
    environment_report = clair.validate(PROJECT, env=environment_name)
    state = "OK" if environment_report.is_valid else f"{environment_report.problem_count} problems"
    print(f"{environment_name:<6} {state}")

dev    OK


prod   OK


## `clair.catalog()` — the documentation data as a dict

`clair docs` starts a web UI. `clair.catalog()` gives the data behind that UI: one entry
for each Trouve, with the docs, the columns, the tests, and the lineage edges.

In [11]:
project_catalog = clair.catalog(PROJECT)

print(sorted(project_catalog))
print(f"{len(project_catalog['trouves'])} Trouves, {len(project_catalog['edges'])} edges")

pd.DataFrame(
    [
        {
            "trouve": address.split(".", 1)[1],
            "type": entry["type"],
            "columns from": entry["column_inference"]["status"],
            "column count": len(entry["column_inference"]["columns"]),
            "tests": len(entry.get("tests") or []),
            "docs": (entry.get("docs") or "").strip().split(".")[0][:44],
        }
        for address, entry in sorted(project_catalog["trouves"].items())
    ]
)

['clair_version', 'edges', 'generated_at', 'project_name', 'trouves']
5 Trouves, 4 edges


,trouve,type,columns from,column count,tests,docs
0,derived.daily_event_counts,table,declared,3,2,Daily count of each event type
1,derived.top_event_types,table,declared,3,0,The 10 event types with the highest total co
2,reference.event_type_labels,table,declared,3,0,The label of each event type
3,refined.events,table,declared,13,0,The refined layer
4,source.events,source,declared,5,0,An events table that exists before clair run


`column_inference.status` tells you where the columns come from. `declared` means that the
author wrote a `Column` list. clair reads the other columns from the SQL.

A catalog is a plain dict, thus a notebook makes a report of it. This example finds each
column name of the project, and the Trouves that hold it. A column with many Trouves is a
join key. A column with one Trouve and a common name is often a duplicate of a join key.

In [12]:
columns_frame = pd.DataFrame(
    [
        {"trouve": address.split(".", 1)[1], "column": column["name"], "type": column["type"]}
        for address, entry in project_catalog["trouves"].items()
        for column in entry["column_inference"]["columns"]
    ]
)

columns_frame.groupby("column")["trouve"].agg(
    trouve_count="count", trouves=", ".join
).sort_values("trouve_count", ascending=False).head(8)

,trouve_count,trouves
column,,
event_type,5,"derived.daily_event_counts, derived.top_event_..."
event_date,2,"derived.daily_event_counts, refined.events"
label,2,"derived.top_event_types, reference.event_type_..."
user_id,2,"refined.events, source.events"
event_id,2,"refined.events, source.events"
occurred_at,2,"refined.events, source.events"
event_count,1,derived.daily_event_counts
element,1,refined.events


### A test coverage report

The catalog also answers "which Trouve has no data quality test". Add this to your
continuous integration, and the coverage does not fall.

In [13]:
untested = sorted(
    address
    for address, entry in project_catalog["trouves"].items()
    if entry["type"] != "source" and not entry.get("tests")
)
tested_count = len(project_catalog["trouves"]) - len(untested)

print(f"{tested_count} of {len(project_catalog['trouves'])} Trouves hold a test.")
for address in untested:
    print("  no test:", address)

2 of 5 Trouves hold a test.
  no test: example_4_database.derived.top_event_types
  no test: example_4_database.reference.event_type_labels
  no test: example_4_database.refined.events


## `clair.clean()` — remove the artifacts of the old runs

Each compile and each run writes a directory under the artifacts directory of the project.
`clair.clean()` removes them. `before` takes `'today'`, `'yesterday'`, `'last_week'`, a
duration such as `'7d'`, or an ISO date. `dry_run=True` names the runs and it removes
nothing.

In [14]:
clean_output = clair.clean(PROJECT, dry_run=True)

print(f"artifacts directory: {clean_output.artifacts_dir}")
print(f"runs on disk: {len(clean_output.runs)}")
for artifact_run in clean_output.runs[:5]:
    print(f"  {artifact_run.run_id}")

artifacts directory: /Users/Omer/Documents/Nerd/Rivage/clair/.claude/worktrees/notebooks-refresh/examples/projects/example_4/_clairtifacts
runs on disk: 10
  01a04a8ff2567c57b53b6020a457bf5f
  01a04a8ff3777b82a6f69723cf16d8c3
  01a04a8ff468702786c68575f2c01c20
  01a04a8ff5627ea3a322329723564c13
  01a04a8ff67c75948fcdd10efd5c874e


In [15]:
# Keep the checkout clean: remove each artifact directory that this notebook wrote,
# and remove the temporary environments file.
import shutil

removed = clair.clean(PROJECT)
shutil.rmtree(NOTEBOOK_DIRECTORY)
print(f"removed {len(removed.runs)} run directories and the temporary environments file")

removed 10 run directories and the temporary environments file


## Next

- [02_lineage_and_impact.ipynb](02_lineage_and_impact.ipynb) — read the DAG, draw it, and
  answer an impact question.
- [03_run_without_snowflake.ipynb](03_run_without_snowflake.ipynb) — run a complete
  project against an adapter that you write.
- [04_author_trouves.ipynb](04_author_trouves.ipynb) — write a project in the notebook,
  and test a pandas transform with no warehouse.